# Fase 2 – Entrenar YOLO para detectar naranjas

**Proyecto:** clasificador de fruta por calibre (Taller Integrador II + Inteligencia Artificial).

El modelo aprende a **encontrar el contorno de cada naranja** (segmentación, una sola clase: `naranja`).
El calibre (chico / mediano / grande) lo sigue calculando `clasificador_naranjas.py` a partir de ese contorno,
con la calibración en mm y las correcciones de perspectiva.

**Pasos:**
1. En el Mac, dentro de la carpeta `Clasificador`, ejecutar `python3 generar_dataset.py`.
2. Revisar las imágenes de `dataset_naranjas/revision` (que los contornos estén bien).
3. Comprimir la carpeta `dataset_naranjas` como `dataset_naranjas.zip`.
4. En Colab: **Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)** y ejecutar las celdas en orden.

## 1. Revisar que haya GPU

In [ ]:
!nvidia-smi

## 2. Instalar YOLO (librería Ultralytics)

In [ ]:
!pip install -q ultralytics
import ultralytics
ultralytics.checks()

## 3. Subir el dataset
Selecciona el archivo `dataset_naranjas.zip`.

In [ ]:
from google.colab import files
import zipfile, os

subido = files.upload()                      # abre el selector de archivos
nombre_zip = list(subido.keys())[0]
with zipfile.ZipFile(nombre_zip) as z:
    z.extractall("/content")

# El zip puede traer la carpeta dataset_naranjas adentro o directamente su contenido
if os.path.exists("/content/dataset_naranjas/data.yaml"):
    CARPETA = "/content/dataset_naranjas"
else:
    CARPETA = "/content"
print("Dataset en:", CARPETA)
print("Fotos de entrenamiento:", len(os.listdir(CARPETA + "/images/train")))
print("Fotos de validación:  ", len(os.listdir(CARPETA + "/images/val")))

## 4. Ajustar la ruta del dataset en `data.yaml`

In [ ]:
with open(CARPETA + "/data.yaml", "w") as f:
    f.write(f"path: {CARPETA}\n")
    f.write("train: images/train\n")
    f.write("val: images/val\n")
    f.write("names:\n")
    f.write("  0: naranja\n")
print(open(CARPETA + "/data.yaml").read())

## 5. Entrenar

- `yolo26n-seg.pt`: YOLO26, versión **nano** de segmentación: la más liviana, pensada para equipos pequeños como una Raspberry Pi.
- Se parte de un modelo ya entrenado con miles de imágenes (*transfer learning*), así que con pocas fotos ya aprende.
- `epochs`: cuántas veces recorre todas las fotos. Con una GPU T4 y ~100 fotos tarda unos minutos.
- YOLO aplica automáticamente "aumento de datos" (giros, espejos, cambios de brillo), que multiplica la variedad de las fotos.

In [ ]:
from ultralytics import YOLO

modelo = YOLO("yolo26n-seg.pt")
resultados = modelo.train(
    data=CARPETA + "/data.yaml",
    epochs=100,
    imgsz=640,
    batch=8,
    patience=0,        # sin detención temprana: con pocas fotos de validación puede cortar demasiado pronto
    flipud=0.5,        # la cámara mira desde arriba: la fruta puede aparecer "de cabeza"
    degrees=180,       # y en cualquier ángulo
    project="/content/runs",
    name="naranjas",
)

## 6. Ver los resultados

Lo más importante es **mAP50-95 (M)** (calidad de las máscaras): mientras más cerca de 1, mejor.

In [ ]:
from IPython.display import Image, display
ruta = "/content/runs/naranjas"
display(Image(ruta + "/results.png", width=900))
display(Image(ruta + "/val_batch0_pred.jpg", width=700))

## 7. Probar el modelo con las fotos de validación

In [ ]:
mejor = YOLO("/content/runs/naranjas/weights/best.pt")
pruebas = mejor.predict(CARPETA + "/images/val", conf=0.7, save=True,
                        project="/content/runs", name="prueba", exist_ok=True)
for r in pruebas:
    print(os.path.basename(r.path), "->", len(r.boxes), "naranjas")

import glob
for img in sorted(glob.glob("/content/runs/prueba/*.jpg")):
    display(Image(img, width=600))

## 8. Descargar el modelo

- `best.pt`: el modelo entrenado (para usarlo en el Mac).
- Formato NCNN (opcional): versión optimizada para la Raspberry Pi.

In [ ]:
files.download("/content/runs/naranjas/weights/best.pt")

In [ ]:
# Opcional: exportar para Raspberry Pi
!pip install -q ncnn pnnx
mejor.export(format="ncnn", imgsz=640)
!cd /content/runs/naranjas/weights && zip -qr best_ncnn_model.zip best_ncnn_model
files.download("/content/runs/naranjas/weights/best_ncnn_model.zip")